<a href="https://colab.research.google.com/github/cahayajatrameylani-glitch/cahayajatrameylani-glitch.github.io/blob/main/Welcome_to_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip -q install transformers accelerate gradio

In [6]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

SYSTEM = """You are a discrete mathematics tutor.
Solve the question step by step:
1. State what is being asked and the topic.
2. Show each step with the formula or rule used.
3. Double-check arithmetic.
4. End with a line starting 'Final answer:'."""

def solve(question):
    if not question.strip():
        return "Please type a question."

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False
        )

    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

demo = gr.Interface(
    fn=solve,
    inputs=gr.Textbox(
        lines=3,
        label="Discrete math question"
    ),
    outputs=gr.Markdown(label="Answer"),
    title="Discrete Math Helper",
    examples=[
        "How many ways can 5 people sit around a circular table?",
        "Find the number of edges in K_8.",
        "How many 3-letter strings can be made from A-E with no repeats?"
    ]
)

demo.launch(share=True)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://98ff014c46f6bdecb9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
